# DIGER Proxy Baseline On MASI CSJ

This notebook runs a documented DIGER-style proxy on the same Amazon Reviews 2023 Clothing/Shoes/Jewelry split contract used by MASI.

Important: this is not an exact DIGER reproduction. DIGER jointly optimizes semantic IDs and the recommender with differentiable Gumbel-based indexing and uncertainty decay. That end-to-end tokenizer/recommender coupling is not implemented in this repository yet. This notebook keeps semantic IDs fixed and disables MASI's cross-modal MLM, giving a static two-stage SID control on the same data and metrics.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
REPO_BRANCH = "main"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
RUNNING_ON_KAGGLE = KAGGLE_WORKING_ROOT.exists() and KAGGLE_INPUT_ROOT.exists()
USE_GIT_CLONE_ON_KAGGLE = True
KAGGLE_REPO_DIR = KAGGLE_WORKING_ROOT / "MASI"
STORAGE_ROOT = KAGGLE_WORKING_ROOT / "masi_artifacts" if RUNNING_ON_KAGGLE else None
BASELINE_PROFILE = "long_safe"  # "smoke_safe" for a quick validation run
RUN_PIP_INSTALL = True
REQUIRE_MASI_TOKEN_ARTIFACT = False

PROFILES = {
    "smoke_safe": {
        "dataset": {"max_users": 256, "max_items": 512, "max_review_records": 150000},
        "experiment": {"autoregressive_epochs": 2, "batch_size": 16, "max_eval_candidates": 512},
    },
    "long_safe": {
        "dataset": {"max_users": 12288, "max_items": 24576, "max_review_records": 50000000},
        "experiment": {"autoregressive_epochs": 30, "batch_size": 32, "max_eval_candidates": 1024},
    },
}


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "masi").exists():
            return candidate
    raise FileNotFoundError("Could not find the MASI repository root.")


if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE:
    KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(KAGGLE_WORKING_ROOT)
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(KAGGLE_REPO_DIR)],
        check=True,
        cwd=KAGGLE_WORKING_ROOT,
    )
    REPO_DIR = KAGGLE_REPO_DIR
else:
    REPO_DIR = find_repo_root(Path.cwd())

if STORAGE_ROOT is None:
    STORAGE_ROOT = REPO_DIR
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
print(f"Repo: {REPO_DIR}")
print(f"Storage: {STORAGE_ROOT}")

In [ ]:
if RUN_PIP_INSTALL:
    pip_base = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    subprocess.run([*pip_base, "--upgrade", "pip"], check=True)
    subprocess.run([*pip_base, "numpy>=1.26,<2.1"], check=True)
    subprocess.run([*pip_base, "-e", ".[recommender]"], check=True)
    print("Packages ready.")
else:
    print("Skipping package installation.")

In [ ]:
def find_prepared_reviews() -> Path | None:
    if not RUNNING_ON_KAGGLE:
        local = REPO_DIR / "data" / "full_dataset" / "Clothing_Shoes_and_Jewelry.jsonl"
        fallback = REPO_DIR / "data" / "raw" / "amazon_reviews_2023" / "Clothing_Shoes_and_Jewelry.jsonl"
        return local if local.exists() else (fallback if fallback.exists() else None)
    candidates = [
        KAGGLE_INPUT_ROOT / "datasets" / "dheerajrajanala" / "masi-amazon-csj-full-dataset" / "Clothing_Shoes_and_Jewelry.jsonl",
        KAGGLE_INPUT_ROOT / "masi-amazon-csj-full-dataset" / "Clothing_Shoes_and_Jewelry.jsonl",
    ]
    candidates.extend(KAGGLE_INPUT_ROOT.rglob("Clothing_Shoes_and_Jewelry.jsonl"))
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


def find_masi_token_artifact() -> Path | None:
    env_path = os.environ.get("MASI_TOKEN_ARTIFACT_PATH")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    candidates.extend([
        STORAGE_ROOT / "outputs" / "amazon_csj_full_dataset_train" / "phase12_tokens" / "fused_semantic_ids.jsonl",
        STORAGE_ROOT / "outputs" / "amazon_csj_full_dataset_kaggle_long_safe_train" / "phase12_tokens" / "fused_semantic_ids.jsonl",
        REPO_DIR / "outputs" / "amazon_csj_full_dataset_train" / "phase12_tokens" / "fused_semantic_ids.jsonl",
        REPO_DIR / "outputs" / "amazon_csj_full_dataset_kaggle_long_safe_train" / "phase12_tokens" / "fused_semantic_ids.jsonl",
    ])
    for root in [STORAGE_ROOT / "outputs", REPO_DIR / "outputs"]:
        if root.exists():
            candidates.extend(sorted(root.glob("*/phase12_tokens/fused_semantic_ids.jsonl")))
    if RUNNING_ON_KAGGLE:
        candidates.extend(KAGGLE_INPUT_ROOT.rglob("fused_semantic_ids.jsonl"))
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


profile = PROFILES[BASELINE_PROFILE]
base_config = json.loads((REPO_DIR / "configs" / "baseline_diger_proxy_full_dataset.json").read_text(encoding="utf-8"))
config = deepcopy(base_config)
config["dataset"].update(profile["dataset"])
config.update(profile["experiment"])

reviews_path = find_prepared_reviews()
if reviews_path is None:
    raise FileNotFoundError("Could not find Clothing_Shoes_and_Jewelry.jsonl. Attach the prepared MASI CSJ dataset or prepare data/full_dataset locally.")
config["dataset"]["reviews_path"] = str(reviews_path)

token_artifact = find_masi_token_artifact()
if token_artifact is not None:
    config["token_artifact_path"] = str(token_artifact)
    config["require_token_artifact"] = True
else:
    config.pop("token_artifact_path", None)
    config["require_token_artifact"] = bool(REQUIRE_MASI_TOKEN_ARTIFACT)
    print("No MASI fused_semantic_ids.jsonl found; this run will use the repository's review-field proxy token fallback unless REQUIRE_MASI_TOKEN_ARTIFACT is True.")

run_root = STORAGE_ROOT / "outputs" / "baseline_diger_proxy_full_dataset"
config["outputs_root"] = str(run_root)
config["checkpoint_root"] = str(run_root / "checkpoints")

resolved_config_dir = STORAGE_ROOT / "resolved_configs"
resolved_config_dir.mkdir(parents=True, exist_ok=True)
resolved_config_path = resolved_config_dir / "baseline_diger_proxy_full_dataset.json"
resolved_config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Using reviews: {reviews_path}")
print(f"Using tokens: {config.get('token_artifact_path')}")
print(f"Resolved config: {resolved_config_path}")

In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "run_masi_experiment.py"), "--config", str(resolved_config_path)],
    check=True,
    cwd=REPO_DIR,
    env=env,
)

In [ ]:
summary_path = Path(config["outputs_root"]) / "experiment_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"Summary: {summary_path}")
print(json.dumps({
    "baseline": config["baseline_note"]["name"],
    "token_source": summary["import_summary"].get("token_source"),
    "num_items": summary["num_items"],
    "num_train_examples": summary["num_train_examples"],
    "warm_metrics": summary["warm_metrics"],
    "cold_metrics": summary["cold_metrics"],
}, indent=2))